## Task 3: Cleaning and analysing data using NumPy

For Task 3, we are writing two NumPy functions that are used in the full workflow in Task 5.

1. `remove_outliers` - finds hardware glitch prices (such as `999.00`) and replaces them with the median of the valid prices.
2. `compute_refund_ratio` - calculates the refund ratio for each item from the number of sales and refunds.

Both functions work on whole NumPy arrays at once (vectorised operations), instead of looping over each value in Python.

---

### 3.1 Removing price outliers

The function takes the price data and converts it to a 1D NumPy float array, so it accepts a list, a NumPy array or a pandas column. It then:

* Calculates the mean ($\mu$) and the standard deviation ($\sigma$) of all the prices.
* Marks a price as an outlier if it deviates from the mean by more than three standard deviations: $\lvert x - \mu \rvert > 3\sigma$. This gives a boolean mask that is `True` for every outlier.
* Calculates the median of the valid prices (the ones that are not outliers) and puts it in place of each outlier.

The function works on a copy, so the original input is not changed.

In [29]:
import numpy as np
import pandas as pd
def remove_outliers(prices: list[float] | np.ndarray | pd.Series) -> np.ndarray:
    """
    Remove outliers from a list of prices using the formula described. (|x - mu| > 3 * sigma)

    Parameters
    ----------
    prices : list[float] | np.ndarray | pd.Series
        A list, NumPy array, or Pandas series of price values.

    Returns
    -------
    np.ndarray
        An array with outliers replaced by the median.
    """
    arr = np.array(prices, dtype=float).flatten()
    
    mu = arr.mean()
    sigma = arr.std()

    outliers = np.abs(arr - mu) > 3 * sigma

    median = np.median(arr[~outliers])
    arr[outliers] = median

    return arr


---

### 3.2 Computing refund ratios

The function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` takes two arrays of the same length: the number of sales and the number of refunds for each item. The ratio is calculated element-wise:

$$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$

* NumPy divides the arrays element by element, so the first refund count is divided by the first sales count, and so on. No loop is needed.
* The small number $\epsilon = 10^{-6}$ is added to the sales count so we never divide by zero, for example if an item has no sales. It is too small to noticeably change the ratio for items that do have sales.

The function returns an array of ratios in the same order as the input arrays. It is used in Task 4, where the counts are taken from the DataFrame.

In [30]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Compute the refund ratio for each product.

    Parameters
    ----------
    sales_count : np.ndarray
        An array of sales counts for each product.
    refund_count : np.ndarray
        An array of refund counts for each product.

    Returns
    -------
    np.ndarray
        An array of refund ratios for each product.
    """

    refund_ratio = refund_count / (sales_count + 1e-6)

    return refund_ratio


---

### Testing each function

We test both functions on small example arrays where the correct result is easy to check by hand.

In [31]:
# Test remove_outliers: 12 normal prices and one glitch (999.0)
test_prices: list[float] = [4.5, 3.0, 5.0, 4.0, 3.5, 4.5, 5.5, 3.0, 4.0, 4.5, 5.0, 3.5, 999.0]
cleaned: np.ndarray = remove_outliers(test_prices)

print("Original prices:", test_prices)
print("Cleaned prices: ", cleaned)

# Test compute_refund_ratio: the last item has no sales, so epsilon prevents division by zero
sales: np.ndarray = np.array([10, 4, 0])
refunds: np.ndarray = np.array([2, 1, 0])
ratios: np.ndarray = compute_refund_ratio(sales, refunds)

print("\nRefund ratios:", np.round(ratios, 4))

# Edge case: refunds but no sales gives a very large ratio
print("0 sales, 1 refund:", compute_refund_ratio(np.array([0]), np.array([1])))

Original prices: [4.5, 3.0, 5.0, 4.0, 3.5, 4.5, 5.5, 3.0, 4.0, 4.5, 5.0, 3.5, 999.0]
Cleaned prices:  [4.5  3.   5.   4.   3.5  4.5  5.5  3.   4.   4.5  5.   3.5  4.25]

Refund ratios: [0.2  0.25 0.  ]
0 sales, 1 refund: [1000000.]


---

### Results and discussion of Task 3

`remove_outliers` replaced the glitch price `999.0` with `4.25`, which is the median of the 12 valid prices. The other prices kept their values and positions, and the original list was not changed, since the function works on a copy.

`compute_refund_ratio` returned `[0.2, 0.25, 0.0]` for the three test items. Because of $\epsilon$, the unrounded values are very slightly smaller than the exact ratios (for example `0.19999998` instead of `0.2`), which makes no practical difference.

---

## Task 4: Data analysis using Pandas

For Task 4, we are writing four Pandas functions that are used in the full workflow in Task 5.

1. `build_dataframe` - converts the list of transaction dictionaries from Task 1 into a DataFrame, with the timestamp as a sorted `datetime` index.
2. `hourly_revenue` - groups the transactions by item and sums the price in 1-hour windows.
3. `status_pivot` - creates a pivot table with the total revenue for each item across the status categories `SUCCESS`, `REFUND` and `CANCELLED`.
4. `refund_ratios` - counts the sales and refunds for each item and calculates the refund ratio using `compute_refund_ratio` from Task 3.

The last three functions all take the DataFrame from `build_dataframe` as input, so they can be called one after another in Task 5.

---

### 4.1 Building the DataFrame

The function takes the list of dictionaries returned by the log parser in Task 1, where each dictionary is one transaction. It then:

* Creates a `pandas.DataFrame`, where each dictionary becomes a row and each key becomes a column.
* Converts the `timestamp` column from text to `datetime` using `pd.to_datetime()`.
* Sets `timestamp` as the index of the DataFrame and sorts the rows chronologically.

A sorted `datetime` index is what makes it possible to resample the transactions by time in Task 4.2.

In [32]:
def build_dataframe(transactions: list[dict[str, str | float]]) -> pd.DataFrame:
    """
    (docstring)
    """
    df = pd.DataFrame(transactions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    # Converting to float just in case
    df["price"] = df["price"].astype(float)
    
    df = df.set_index("timestamp").sort_index()
    
    return df

---

### 4.2 Hourly revenue per item

The function groups the transactions by `item`, and then resamples each group into 1-hour windows (`"1h"`), summing the `price` column within each window. The result has one total for each combination of item and hour.

We sum the price of all transactions, regardless of status. This follows the assignment, which also counts `REFUND` and `CANCELLED` transactions as "transaction revenue" in Task 4.3.

In [33]:
def hourely_sales(df: pd.DataFrame) -> pd.DataFrame:
    """
    (docstring)
    """
    return df.groupby("item")["price"].resample("1h").sum().to_frame(name="price")

---

### 4.3 Pivot table of revenue by status

The function uses `pivot_table()` to create a table with one row per item and one column per status (`SUCCESS`, `REFUND` and `CANCELLED`). Each cell contains the total price of the transactions with that item and status.

If an item has no transactions with a given status, the cell is filled with `0` instead of being left empty (`NaN`).

---

### 4.4 Refund ratio per item

The function counts the number of transactions for each combination of item and status using `pd.crosstab()`. It then:

* Uses the number of `SUCCESS` transactions as the sales count, and the number of `REFUND` transactions as the refund count.
* Converts both columns to NumPy arrays and passes them to `compute_refund_ratio` from Task 3.
* Returns a DataFrame with the item, sales count, refund count and refund ratio.

If a status never appears in the data, its column is added with zeros, so the function still works.

---

### Testing each function

We test all four functions on the transactions parsed from the log files generated by `generate_data_for_d2.py`.

---

### Results and discussion of Task 4

<!-- Write this after running the tests above -->